In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import butter, sosfiltfilt, lfilter, welch, coherence, csd, correlate, correlation_lags, freqz
from scipy import stats

trapz = getattr(np, "trapezoid", None) or np.trapz

In [ ]:
files = {
    "Sidewalk": "imu_accel_sidewalk_cafe.csv",
    "Asphalt":  "imu_accel_cafe_asphalt.csv",
}
surface = {"Sidewalk": "Sidewalk", "Asphalt": "Asphalt"}
calib_files = {}

vr_files = {}
vr_field = {}
vr_input = {}

out_dir = Path("validation_outputs")
out_dir.mkdir(exist_ok = True)

In [ ]:
fs = 200.0
max_gap = 0.05
min_segment = 5.0

gravity_fc = 0.3
band = (0.5, 80.0)
bp_order = 4
q, r = 1e-4, 1e-2
freq_bands = {"low": (0.5, 5.0), "mid": (5.0, 20.0), "high": (20.0, 80.0)}
nperseg = int(2.0 * fs)

motion_win = 1.0
motion_thresh = None
manual_exclude = {}

event_theta = None
event_sigma = None
event_refractory = 0.25

stat_win = 2.0
min_moving_frac = 0.9

paper_lpf_fc = 0.5
threshold_low, threshold_high = 0.05, 0.15

In [ ]:
datasets = {}
for name, path in files.items():
    df = pd.read_csv(path)
    df = df.sort_values("time_sec").drop_duplicates("time_sec").reset_index(drop = True)
    datasets[name] = df

    dt = df["time_sec"].diff()
    median_dt = dt.median()
    print(f"{name}: {df.shape[0]} rows, {df['time_sec'].iloc[-1]:.1f} s, "
          f"median dt = {median_dt:.5f}s (~{1/median_dt:.1f} Hz)")
    print(f"   gaps > {max_gap}s: {(dt > max_gap).sum()}, largest 3: {dt.nlargest(3).round(4).tolist()}")
    print(f"   NaNs in ax/ay/az: {df[['ax','ay','az']].isna().sum().to_dict()}")
    g = df[["ax", "ay", "az"]].mean().to_numpy()
    print(f"   mean acceleration = {np.round(g, 2)}, |g| = {np.linalg.norm(g):.2f}")

In [ ]:
def rotation_from_calibration(calib_df):
    a = calib_df[["ax", "ay", "az"]].to_numpy(float).mean(axis = 0)
    g = a / np.linalg.norm(a)
    target = np.array([0.0, 0.0, -1.0])
    v = np.cross(g, target)
    c = float(np.dot(g, target))
    if np.isclose(c, 1.0):
        return np.eye(3)
    if np.isclose(c, -1.0):
        axis = np.cross(g, [1, 0, 0]) if abs(g[0]) < 0.9 else np.cross(g, [0, 1, 0])
        axis = axis / np.linalg.norm(axis)
        return 2 * np.outer(axis, axis) - np.eye(3)
    vx = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    return np.eye(3) + vx + vx @ vx * (1 / (1 + c))

def apply_rotation(df, R):
    df = df.copy()
    df[["ax", "ay", "az"]] = df[["ax", "ay", "az"]].to_numpy(float) @ R.T
    return df

rotations = {}
for name in list(files) + list(vr_files):
    if name in calib_files:
        rotations[name] = rotation_from_calibration(pd.read_csv(calib_files[name]))
        print(f"{name}: rotation matrix\n{np.round(rotations[name], 4)}")
    else:
        rotations[name] = np.eye(3)
        print(f"{name}: no calibration file, no rotation")

In [ ]:
def split_and_resample(df):
    t = df["time_sec"].to_numpy(float)
    breaks = np.where(np.diff(t) > max_gap)[0] + 1
    pieces = []
    for seg_id, idx in enumerate(np.split(np.arange(len(t)), breaks)):
        if len(idx) < 2 or t[idx[-1]] - t[idx[0]] < min_segment:
            continue
        tt = np.arange(t[idx[0]], t[idx[-1]], 1 / fs)
        piece = pd.DataFrame({"t": tt, "segment": seg_id})
        for axis in ("ax", "ay", "az"):
            piece[axis] = np.interp(tt, t[idx], df[axis].to_numpy(float)[idx])
        pieces.append(piece)
    return pieces

def gravity_lowpass(x, cutoff_hz = gravity_fc):
    alpha = 1.0 / (1.0 + 2 * np.pi * cutoff_hz / fs)
    return lfilter([1 - alpha], [1, -alpha], x, zi = [alpha * x[0]])[0]

sos_bp = butter(bp_order, band, btype = "bandpass", fs = fs, output = "sos")

def kalman_smooth(z, q = q, r = r):
    z = np.asarray(z, dtype = float)
    xhat = np.zeros(len(z))
    xhat[0] = z[0]
    P = 1.0
    for k in range(1, len(z)):
        P_prior = P + q
        K = P_prior / (P_prior + r)
        xhat[k] = xhat[k-1] + K * (z[k] - xhat[k-1])
        P = (1 - K) * P_prior
    return xhat

def process(df, name):
    df = apply_rotation(df, rotations.get(name, np.eye(3)))
    pieces = []
    for piece in split_and_resample(df):
        for axis in ("ax", "ay", "az"):
            dyn = piece[axis].to_numpy() - gravity_lowpass(piece[axis].to_numpy())
            piece[f"{axis}_dyn"] = dyn
            piece[f"{axis}_bp"] = sosfiltfilt(sos_bp, dyn)
        piece["v"] = np.sqrt(piece["ay_bp"] ** 2 + piece["az_bp"] ** 2)
        piece["v_kalman"] = kalman_smooth(piece["v"].to_numpy())
        pieces.append(piece)
    out = pd.concat(pieces, ignore_index = True)
    out["edge"] = (out["t"] - out.groupby("segment")["t"].transform("min")) < 1.0
    return out

proc = {}
for name, df in datasets.items():
    proc[name] = process(df, name)
    print(f"{name}: {proc[name]['segment'].nunique()} segments, {len(proc[name]) / fs:.1f} s kept")

In [ ]:
def rolling_rms(x, n):
    return np.sqrt(pd.Series(x ** 2).rolling(n, center = True, min_periods = 1).mean().to_numpy())

def otsu_threshold(values, bins = 256):
    hist, edges = np.histogram(values, bins = bins)
    mids = (edges[:-1] + edges[1:]) / 2
    w0 = np.cumsum(hist)
    w1 = w0[-1] - w0
    m0 = np.cumsum(hist * mids) / np.maximum(w0, 1)
    m1 = (np.sum(hist * mids) - np.cumsum(hist * mids)) / np.maximum(w1, 1)
    return mids[np.argmax(w0 * w1 * (m0 - m1) ** 2)]

n_win = int(motion_win * fs)
for name, p in proc.items():
    p["v_rms1s"] = p.groupby("segment")["v"].transform(lambda s: rolling_rms(s.to_numpy(), n_win))

log_rms = np.log10(np.concatenate([p.loc[~p["edge"], "v_rms1s"].to_numpy() for p in proc.values()]) + 1e-6)
if motion_thresh is None:
    motion_thresh = 10 ** otsu_threshold(log_rms)
print(f"motion threshold = {motion_thresh:.3f} m/s^2")

for name, p in proc.items():
    moving = (p["v_rms1s"] > motion_thresh) & ~p["edge"]
    for t0, t1 in manual_exclude.get(name, []):
        moving = moving & ~p["t"].between(t0, t1)
    p["moving"] = moving
    print(f"{name}: moving {moving.mean() * 100:.1f}% ({moving.sum() / fs:.1f} s)")

plt.figure(figsize = (15, 4))
plt.hist(log_rms, bins = 200)
plt.axvline(np.log10(motion_thresh), color = "red")
plt.title("Motion threshold")
plt.xlabel("log10(1-s RMS of v)"); plt.grid(True); plt.show()

for name, p in proc.items():
    plt.figure(figsize = (15, 4))
    plt.plot(p["t"], p["v_rms1s"], linewidth = 0.5, label = "1-s RMS")
    plt.fill_between(p["t"], 0, p["v_rms1s"].max(), where = ~p["moving"], color = "gray", alpha = 0.25, label = "excluded")
    plt.axhline(motion_thresh, color = "red", linewidth = 0.8)
    plt.title(f"Motion mask - {name}")
    plt.xlabel("Time (seconds)"); plt.legend(); plt.grid(True)
    plt.savefig(out_dir / f"motion_mask_{name}.png", dpi = 200, bbox_inches = "tight")
    plt.show()

In [ ]:
def moving_stretches(mask):
    m = np.concatenate([[False], np.asarray(mask, bool), [False]])
    d = np.diff(m.astype(int))
    return list(zip(np.where(d == 1)[0], np.where(d == -1)[0]))

def psd_moving(p, col_suffix = "_bp", axes = ("ay", "az")):
    total, weight, f = None, 0, None
    for seg_id, s in p.groupby("segment"):
        for a, b in moving_stretches(s["moving"].to_numpy()):
            if b - a < nperseg:
                continue
            P_sum = 0
            for axis in axes:
                col = axis + col_suffix if col_suffix else axis
                f, P = welch(s[col].to_numpy()[a:b], fs = fs, nperseg = nperseg)
                P_sum = P_sum + P
            w = (b - a) // (nperseg // 2)
            total = P_sum * w if total is None else total + P_sum * w
            weight += w
    return f, total / weight

def spectral_features(f, P):
    sel = (f >= band[0]) & (f <= band[1])
    feats = {"f_peak": f[sel][np.argmax(P[sel])],
             "f_c": np.sum(f[sel] * P[sel]) / np.sum(P[sel])}
    for b_name, (lo, hi) in freq_bands.items():
        b = (f >= lo) & (f < hi)
        feats[f"band_{b_name}"] = np.sqrt(trapz(P[b], f[b]))
    return feats

psd = {}
for name, p in proc.items():
    psd[name] = psd_moving(p)

In [ ]:
v_moving = np.concatenate([p.loc[p["moving"], "v"].to_numpy() for p in proc.values()])
if event_theta is None:
    event_theta = np.median(v_moving) + 6 * stats.median_abs_deviation(v_moving, scale = "normal")
if event_sigma is None:
    event_sigma = event_theta / 4
print(f"event theta = {event_theta:.2f} m/s^2, sigma = {event_sigma:.2f} m/s^2")

def detect_events(p):
    v = p["v"].to_numpy()
    dv = np.diff(v, prepend = v[0])
    candidates = np.where((v > event_theta) & (dv > event_sigma) & p["moving"].to_numpy())[0]
    t = p["t"].to_numpy()
    events, last_t = [], -np.inf
    for i in candidates:
        if t[i] - last_t >= event_refractory:
            events.append(i)
            last_t = t[i]
    return np.array(events, dtype = int)

events = {}
for name, p in proc.items():
    events[name] = detect_events(p)
    print(f"{name}: {len(events[name])} events")

In [ ]:
def summarize(name, p, df_raw):
    moving = p["moving"].to_numpy()
    moving_s = moving.sum() / fs
    row = {"Run_id": name, "Surface": surface.get(name, name), "moving_s": round(moving_s, 1),
           "a_RMS": np.sqrt(np.mean(p.loc[moving, "v"] ** 2)),
           **spectral_features(*psd[name]),
           "events": len(events[name]), "events_per_s": len(events[name]) / moving_s}
    if "speed_mps" in df_raw.columns:
        speed = np.interp(p["t"], df_raw["time_sec"], df_raw["speed_mps"])
        distance = np.sum(speed[moving]) / fs
        row["distance_m"] = distance
        row["events_per_100m"] = len(events[name]) / distance * 100 if distance > 0 else np.nan
    return row

phase2 = pd.DataFrame([summarize(name, proc[name], datasets[name]) for name in proc])
phase2["check_sqrt_sum_bands"] = np.sqrt(sum(phase2[f"band_{b}"] ** 2 for b in freq_bands))
phase2.to_csv(out_dir / "phase2_output.csv", index = False)
phase2.round(3)

In [ ]:
plt.figure(figsize = (12, 5))
for name, (f, P) in psd.items():
    sel = (f >= band[0]) & (f <= band[1])
    plt.semilogy(f[sel], P[sel], label = name)
for lo, hi in freq_bands.values():
    plt.axvline(lo, color = "gray", linestyle = ":", linewidth = 0.8)
plt.title("Field vibration PSD (moving only)")
plt.xlabel("Frequency (Hz)"); plt.ylabel("PSD ((m/s²)²/Hz)"); plt.legend(); plt.grid(True, which = "both", alpha = 0.3)
plt.savefig(out_dir / "field_psd.png", dpi = 300, bbox_inches = "tight")
plt.show()

for name, p in proc.items():
    plt.figure(figsize = (15, 4))
    plt.plot(p["t"], p["v"], linewidth = 0.3, alpha = 0.6, label = "v (band-limited)")
    plt.plot(p["t"], p["v_kalman"], linewidth = 0.8, label = "Kalman")
    ev = events[name]
    plt.plot(p["t"].to_numpy()[ev], p["v"].to_numpy()[ev], "rv", markersize = 4, label = "events")
    plt.title(f"Vibration - {name}")
    plt.xlabel("Time (seconds)"); plt.legend(); plt.grid(True)
    plt.savefig(out_dir / f"field_time_series_{name}.png", dpi = 200, bbox_inches = "tight")
    plt.show()

In [ ]:
def window_metrics(name, p):
    rows = []
    n = int(stat_win * fs)
    for seg_id, s in p.groupby("segment"):
        for a in range(0, len(s) - n + 1, n):
            w = s.iloc[a:a + n]
            if w["moving"].mean() < min_moving_frac:
                continue
            P_sum = 0
            for axis in ("ay", "az"):
                f, P = welch(w[f"{axis}_bp"].to_numpy(), fs = fs, nperseg = n // 2)
                P_sum = P_sum + P
            row = {"run": name, "surface": surface.get(name, name), "t0": w["t"].iloc[0],
                   "a_RMS": np.sqrt(np.mean(w["v"] ** 2))}
            for b_name, (lo, hi) in freq_bands.items():
                b = (f >= lo) & (f < hi)
                row[f"band_{b_name}"] = np.sqrt(trapz(P_sum[b], f[b]))
            rows.append(row)
    return rows

def cohens_d(x1, x2):
    x1, x2 = np.asarray(x1, float), np.asarray(x2, float)
    n1, n2 = len(x1), len(x2)
    s_pooled = np.sqrt(((n1 - 1) * x1.var(ddof = 1) + (n2 - 1) * x2.var(ddof = 1)) / (n1 + n2 - 2))
    return (x1.mean() - x2.mean()) / s_pooled

def separability(df, group_col = "surface", metrics = ("a_RMS", "band_low", "band_mid", "band_high")):
    g1, g2 = df[group_col].unique()[:2]
    rows = []
    for m in metrics:
        x1 = df.loc[df[group_col] == g1, m]
        x2 = df.loc[df[group_col] == g2, m]
        t = stats.ttest_ind(x1, x2, equal_var = False)
        u = stats.mannwhitneyu(x1, x2)
        rows.append({"metric": m, f"mean_{g1}": x1.mean(), f"mean_{g2}": x2.mean(),
                     f"n_{g1}": len(x1), f"n_{g2}": len(x2),
                     "cohens_d": cohens_d(x1, x2), "welch_t": t.statistic, "p_t": t.pvalue,
                     "p_mannwhitney": u.pvalue})
    return pd.DataFrame(rows)

windows = pd.DataFrame([row for name, p in proc.items() for row in window_metrics(name, p)])
windows.to_csv(out_dir / "field_window_metrics.csv", index = False)
print(f"{len(windows)} windows of {stat_win} s")

sep_field = separability(windows)
sep_field.to_csv(out_dir / "field_separability.csv", index = False)
sep_field.round(4)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize = (16, 4))
groups = list(windows["surface"].unique())
for ax_plot, m in zip(axes, ["a_RMS", "band_low", "band_mid", "band_high"]):
    ax_plot.boxplot([windows.loc[windows["surface"] == g, m] for g in groups], showfliers = False)
    ax_plot.set_xticks(range(1, len(groups) + 1))
    ax_plot.set_xticklabels(groups)
    ax_plot.set_title(m)
    ax_plot.set_ylabel("m/s²")
    ax_plot.grid(True)
plt.tight_layout()
plt.savefig(out_dir / "field_separability.png", dpi = 300, bbox_inches = "tight")
plt.show()

In [ ]:
def paper_c_command(datasets):
    smooth = {}
    for name, df in datasets.items():
        dt = df["time_sec"].diff().median()
        rc = 1.0 / (2 * np.pi * paper_lpf_fc)
        alpha = rc / (rc + dt)
        dyn = {}
        for axis in ("ay", "az"):
            x = df[axis].to_numpy(float)
            dyn[axis] = x - lfilter([1 - alpha], [1, -alpha], x, zi = [alpha * x[0]])[0]
        smooth[name] = kalman_smooth(np.sqrt(dyn["ay"] ** 2 + dyn["az"] ** 2))
    all_values = np.concatenate(list(smooth.values()))
    global_min, global_max = all_values.min(), all_values.max()
    command = {}
    for name, m in smooth.items():
        u = (m - global_min) / (global_max - global_min)
        x = np.clip((u - threshold_low) / (threshold_high - threshold_low), 0, 1)
        command[name] = u * x * x * (3 - 2 * x)
    return command

def band_fractions(f, P):
    energy = {}
    for b_name, (lo, hi) in freq_bands.items():
        b = (f >= lo) & (f < hi)
        energy[b_name] = trapz(P[b], f[b])
    total = sum(energy.values())
    return {b_name: e / total for b_name, e in energy.items()}

command = paper_c_command(datasets)

rows = []
for name, p in proc.items():
    p["command"] = np.interp(p["t"], datasets[name]["time_sec"], command[name])
    f_field, P_field = psd[name]
    f_cmd, P_cmd = psd_moving(p, col_suffix = "", axes = ("command",))
    frac_field = band_fractions(f_field, P_field)
    frac_cmd = band_fractions(f_cmd, P_cmd)
    rows.append({"Run_id": name,
                 **{f"field_frac_{b}": frac_field[b] for b in freq_bands},
                 **{f"command_frac_{b}": frac_cmd[b] for b in freq_bands}})

    sel = (f_field >= band[0]) & (f_field <= band[1])
    plt.figure(figsize = (12, 4.5))
    plt.semilogy(f_field[sel], P_field[sel] / trapz(P_field[sel], f_field[sel]), label = "field vibration (normalized)")
    plt.semilogy(f_cmd[sel], P_cmd[sel] / trapz(P_cmd[sel], f_cmd[sel]), label = "current Unity command (normalized)")
    plt.title(f"Field vs current Unity command - {name}")
    plt.xlabel("Frequency (Hz)"); plt.legend(); plt.grid(True, which = "both", alpha = 0.3)
    plt.savefig(out_dir / f"field_vs_command_psd_{name}.png", dpi = 300, bbox_inches = "tight")
    plt.show()

band_frac = pd.DataFrame(rows)
band_frac.to_csv(out_dir / "field_vs_command_band_fractions.csv", index = False)
band_frac.round(4)

In [ ]:
P_prior_ss = (q + np.sqrt(q ** 2 + 4 * q * r)) / 2
K_ss = P_prior_ss / (P_prior_ss + r)
f_cut = -np.log(1 - K_ss) * fs / (2 * np.pi)
print(f"steady-state Kalman gain K = {K_ss:.4f}, equivalent low-pass ~{f_cut:.2f} Hz")

w, h = freqz([K_ss], [1, -(1 - K_ss)], worN = 2048, fs = fs)
plt.figure(figsize = (10, 4))
plt.semilogx(w[1:], 20 * np.log10(np.abs(h[1:])))
plt.axhline(-3, color = "red", linewidth = 0.8)
plt.title("Kalman filter (steady state) magnitude response")
plt.xlabel("Frequency (Hz)"); plt.ylabel("dB"); plt.grid(True, which = "both", alpha = 0.3)
plt.savefig(out_dir / "kalman_response.png", dpi = 300, bbox_inches = "tight")
plt.show()

In [ ]:
def align_lag(ref, sig, marker_s = 10.0, max_lag_s = 5.0):
    n = int(marker_s * fs)
    a = ref[:n] - np.mean(ref[:n])
    b = sig[:n] - np.mean(sig[:n])
    c = correlate(a, b, mode = "full")
    lags = correlation_lags(len(a), len(b), mode = "full")
    ok = np.abs(lags) <= max_lag_s * fs
    return lags[ok][np.argmax(c[ok])] / fs

def bland_altman(field_values, vr_values):
    field_values, vr_values = np.asarray(field_values, float), np.asarray(vr_values, float)
    diff = vr_values - field_values
    avg = (vr_values + field_values) / 2
    bias = diff.mean()
    sd = diff.std(ddof = 1)
    return bias, bias - 1.96 * sd, bias + 1.96 * sd, diff, avg

def log_spectral_distance(f, P_field, P_vr):
    sel = (f >= band[0]) & (f <= band[1])
    return np.sqrt(np.mean((10 * np.log10(P_field[sel]) - 10 * np.log10(P_vr[sel])) ** 2))

def band_ratios(f, P_field, P_vr):
    ratios = {}
    for b_name, (lo, hi) in freq_bands.items():
        b = (f >= lo) & (f < hi)
        ratios[f"ratio_{b_name}"] = trapz(P_vr[b], f[b]) / trapz(P_field[b], f[b])
    return ratios

def transfer_function(x_in, y_out):
    f, Pxy = csd(x_in, y_out, fs = fs, nperseg = nperseg)
    _, Pxx = welch(x_in, fs = fs, nperseg = nperseg)
    return f, Pxy / Pxx

In [ ]:
if not vr_files:
    print("vr_files is empty, Phase 4 comparison skipped")
else:
    vr_datasets, vr_proc = {}, {}
    for name, path in vr_files.items():
        df = pd.read_csv(path)
        vr_datasets[name] = df.sort_values("time_sec").drop_duplicates("time_sec").reset_index(drop = True)
        p = process(vr_datasets[name], name)
        field_p = proc[vr_field[name]]
        lag = align_lag(field_p["v"].to_numpy(), p["v"].to_numpy())
        field_moving = np.interp(p["t"] + lag, field_p["t"], field_p["moving"].astype(float), left = 0, right = 0) > 0.5
        p["moving"] = field_moving & ~p["edge"]
        p["lag_s"] = lag
        vr_proc[name] = p
        psd[name] = psd_moving(p)
        events[name] = detect_events(p)
        surface.setdefault(name, surface[vr_field[name]])

    vr_table = pd.DataFrame([summarize(name, vr_proc[name], vr_datasets[name]) for name in vr_proc])
    vr_table.to_csv(out_dir / "phase2_output_vr.csv", index = False)

    rows, coh_curves, H_curves = [], {}, {}
    field_idx = phase2.set_index("Run_id")
    vr_idx = vr_table.set_index("Run_id")
    for name, p in vr_proc.items():
        field_name = vr_field[name]
        field_p = proc[field_name]
        lag = p["lag_s"].iloc[0]
        f, P_field = psd[field_name]
        _, P_vr = psd[name]

        moving = p["moving"].to_numpy()
        coh_sum = 0
        for axis in ("ay", "az"):
            x_field = np.interp(p["t"] + lag, field_p["t"], field_p[f"{axis}_bp"])[moving]
            f_coh, g2 = coherence(x_field, p[f"{axis}_bp"].to_numpy()[moving], fs = fs, nperseg = nperseg)
            coh_sum = coh_sum + g2 / 2
        coh_curves[name] = (f_coh, coh_sum)

        row = {"VR_run": name, "Field_run": field_name, "Surface": surface[name], "lag_s": lag,
               "G": vr_idx.loc[name, "a_RMS"] / field_idx.loc[field_name, "a_RMS"],
               "centroid_err_Hz": vr_idx.loc[name, "f_c"] - field_idx.loc[field_name, "f_c"],
               "LSD_dB": log_spectral_distance(f, P_field, P_vr),
               **band_ratios(f, P_field, P_vr)}
        for b_name, (lo, hi) in freq_bands.items():
            row[f"coh_{b_name}"] = coh_sum[(f_coh >= lo) & (f_coh < hi)].mean()
        rows.append(row)

        if name in vr_input:
            p_in = vr_proc[vr_input[name]]
            n = min(len(p_in), len(p))
            H_curves[f"{vr_input[name]} -> {name}"] = transfer_function(p_in["az_bp"].to_numpy()[:n], p["az_bp"].to_numpy()[:n])

    comparison = pd.DataFrame(rows)
    comparison.to_csv(out_dir / "phase4_field_vs_vr.csv", index = False)
    print(comparison.round(3).to_string())

In [ ]:
if vr_files:
    bias, loa_low, loa_high, diff, avg = bland_altman(field_idx.loc[comparison["Field_run"], "a_RMS"],
                                                      vr_idx.loc[comparison["VR_run"], "a_RMS"])
    print(f"Bland-Altman a_RMS: bias = {bias:.3f}, limits of agreement = [{loa_low:.3f}, {loa_high:.3f}] m/s^2")
    plt.figure(figsize = (6, 4.5))
    plt.scatter(avg, diff)
    for y in (bias, loa_low, loa_high):
        plt.axhline(y, linestyle = "--", color = "gray")
    plt.title("Bland-Altman (a_RMS)")
    plt.xlabel("Average of field and VR"); plt.ylabel("VR - field"); plt.grid(True)
    plt.savefig(out_dir / "bland_altman.png", dpi = 300, bbox_inches = "tight")
    plt.show()

    fig, axes = plt.subplots(1, 3, figsize = (18, 4.5))
    for name in vr_proc:
        f, P = psd[vr_field[name]]
        sel = (f >= band[0]) & (f <= band[1])
        axes[0].semilogy(f[sel], P[sel], color = "black", linewidth = 0.6, alpha = 0.5)
        f, P = psd[name]
        axes[0].semilogy(f[sel], P[sel], linewidth = 0.8, label = name)
        f_coh, g2 = coh_curves[name]
        axes[1].plot(f_coh[sel], g2[sel], linewidth = 0.8, label = name)
    for label, (f_h, H) in H_curves.items():
        sel = (f_h >= band[0]) & (f_h <= band[1])
        axes[2].semilogy(f_h[sel], np.abs(H[sel]), label = label)
    axes[0].set_title("PSD: field (black) vs VR")
    axes[1].set_title("Coherence field-VR")
    axes[1].set_ylim(0, 1)
    axes[2].set_title("|H(f)| between layers")
    for ax_plot in axes:
        ax_plot.set_xlabel("Frequency (Hz)")
        ax_plot.grid(True, which = "both", alpha = 0.3)
        if ax_plot.get_legend_handles_labels()[0]:
            ax_plot.legend(fontsize = 7)
    plt.tight_layout()
    plt.savefig(out_dir / "phase4_spectra.png", dpi = 300, bbox_inches = "tight")
    plt.show()

In [ ]:
if vr_files:
    import statsmodels.formula.api as smf
    from statsmodels.stats.anova import anova_lm

    both = pd.concat([phase2.assign(environment = "field"), vr_table.assign(environment = "vr")], ignore_index = True)
    both = both.rename(columns = {"Surface": "surface"})
    for env, d in both.groupby("environment"):
        if d["surface"].nunique() == 2 and d.groupby("surface").size().min() >= 2:
            print(f"--- separability, {env} ---")
            print(separability(d).round(4).to_string())

    model = smf.ols("a_RMS ~ C(surface) * C(environment)", data = both).fit()
    print(anova_lm(model, typ = 2))